In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

In [2]:
from src.models.Autoencoder import ResNet34AutoEncoder

resnet34 = ResNet34AutoEncoder()

In [3]:
print(resnet34.model.encoder)

ResNetEncoder(
  (conv1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (conv2): EncoderResidualBlock(
    (00 MaxPooling): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (01 EncoderLayer): EncoderResidualLayer(
      (weight_layer1): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (weight_layer2): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (relu): Sequential(
        (0): ReLU(inplace=True)
      )
    )
    (02 EncoderLayer): 

In [4]:
from src.models.Scaler import CIFAR10Upscaler, CIFAR10Downscaler


class CombinedModel(nn.Module):
    
    def __init__(self, pretrained_model):
        super(CombinedModel, self).__init__()
        self.upscaler = CIFAR10Upscaler()
        self.downscaler = CIFAR10Downscaler()
        self.pretrained_model = pretrained_model
        
    def forward(self, x):
        x = self.upscaler(x)
        x = self.pretrained_model(x)
        x = self.downscaler(x)
        return x

In [3]:
class ResizeTransform:
    def __call__(self, img):
        # Convert image to tensor if it isn't already
        img = transforms.ToTensor()(img)
        # Resize image to (224, 224) using bicubic interpolation
        img = nn.functional.interpolate(img.unsqueeze(0), size=(224, 224), mode='bicubic', align_corners=False)
        return img.squeeze(0)  # Remove the extra batch dimension

In [16]:
dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)
cats_dataset = [(img, label) for (img, label) in dataset if label == 3]
cats_dataloader = torch.utils.data.DataLoader(cats_dataset, batch_size=32, shuffle=True)

Files already downloaded and verified


In [4]:
class CatDataset(torch.utils.data.Dataset):
    def __init__(self, transform=None):
        trainset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True)
        cat_indices = [i for i, label in enumerate(trainset.targets) if label == 3]
        cat_subset = torch.utils.data.Subset(trainset, cat_indices)
        self.subset = cat_subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label 
    
    
# Combine transformations including normalization
transform = transforms.Compose([
    ResizeTransform()
])
cats_dataset = CatDataset(transform)

Files already downloaded and verified


In [6]:
from torch import optim

combined_model = CombinedModel(resnet34)
optimizer = optim.Adam([param for param in combined_model.parameters() if param.requires_grad], lr=0.001)
criterion = nn.MSELoss()

for epoch in range(5):  # Few epochs to adapt the preprocessor layer
    for images, _ in cats_dataloader:
        optimizer.zero_grad()
        outputs = combined_model(images)
        loss = criterion(outputs, images)
        print("Loss: ", loss.item())
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Loss:  0.23865145444869995
Loss:  0.22563159465789795
Loss:  0.18220768868923187
Loss:  0.23038971424102783
Loss:  0.20511572062969208
Loss:  0.2233051061630249
Loss:  0.1811150163412094
Loss:  0.17747698724269867
Loss:  0.20395539700984955
Loss:  0.1578233242034912
Loss:  0.20571203529834747
Loss:  0.21237067878246307
Loss:  0.17083615064620972
Loss:  0.19573140144348145
Loss:  0.15365241467952728
Loss:  0.15435127913951874
Loss:  0.17795288562774658
Loss:  0.16185273230075836
Loss:  0.1497626006603241
Loss:  0.14864885807037354
Loss:  0.16116221249103546
Loss:  0.13929836452007294
Loss:  0.1541169136762619
Loss:  0.10161735862493515
Loss:  0.14504337310791016
Loss:  0.10526936501264572
Loss:  0.11214866489171982
Loss:  0.10653796046972275
Loss:  0.08243484050035477
Loss:  0.09670942276716232
Loss:  0.08482629060745239
Loss:  0.09368440508842468
Loss:  0.08343997597694397
Loss:  0.0713927149772644
Loss:  0.08940675109624863
Loss:  0.06289559602737427
Loss:  0.07264335453510284
Loss:  

KeyboardInterrupt: 

In [ ]:
from src.vmmd import VMMD

vmmd = VMMD(batch_size=10)
vmmd.fit(cats_dataset, resnet34.model.encoder)